# Overlapping Communities: BigCLAM

## Learning Objectives

1. **Define** the Affiliation Graph Model (AGM) for overlapping communities
2. **Derive** the BigCLAM objective (maximum likelihood of AGM)
3. **Implement** a simplified BigCLAM gradient ascent
4. **Compare** overlapping vs hard-partition community detection

## Why Overlapping?

In real networks, nodes belong to **multiple communities simultaneously**:
- A person is in family, work, hobby, and neighborhood communities
- A protein participates in multiple functional complexes
- A web page covers multiple topics

**Hard partitioning** (k-means, Louvain, spectral) forces each node into exactly one community.
**Overlapping** methods assign fractional membership to multiple communities.

## Affiliation Graph Model (AGM)

**Parameters:**
- $k$ communities $C_1, \ldots, C_k$
- Membership strength $F_{uc} \geq 0$: how strongly node $u$ belongs to community $c$
- Community strength $p_c \in (0,1)$: density parameter for community $c$

**Edge probability:**
$$P[(u,v) \in E] = 1 - \exp\left(-\sum_c F_{uc} \cdot F_{vc}\right)$$

**Intuition:** two nodes are connected if they share memberships in any community; probabilities from multiple communities combine.

## BigCLAM: Maximum Likelihood Fitting

**Objective:** find $F$ (the $n \times k$ membership matrix) that maximizes log-likelihood:

$$\ell(F) = \sum_{(u,v)\in E} \log\left(1 - e^{-\mathbf{f}_u^\top \mathbf{f}_v}\right) - \sum_{(u,v)\notin E} \mathbf{f}_u^\top \mathbf{f}_v$$

**Gradient for node $u$, community $c$:**
$$\frac{\partial \ell}{\partial F_{uc}} = \sum_{v\in N(u)} F_{vc} \cdot \frac{e^{-\mathbf{f}_u^\top \mathbf{f}_v}}{1-e^{-\mathbf{f}_u^\top \mathbf{f}_v}} - \sum_{v\notin N(u)} F_{vc}$$

**Key efficiency trick:** the second sum over non-neighbors can be rewritten as:
$$\left(\sum_{v} F_{vc}\right) - F_{uc} - \sum_{v\in N(u)} F_{vc}$$

In [ ]:
import numpy as np
import random

def bigclam(adj, k, n_iter=100, lr=0.01, seed=42):
    """
    Simplified BigCLAM: gradient ascent on membership matrix F.
    adj: dict node->set of neighbors
    k: number of communities
    Returns: F matrix (n x k), non-negative
    """
    rng = np.random.default_rng(seed)
    nodes = sorted(adj.keys())
    n = len(nodes); idx = {v:i for i,v in enumerate(nodes)}
    F = rng.random((n, k)) * 0.5 + 0.1   # initialize positive

    col_sums = F.sum(axis=0)   # sum_v F_{vc} for each c

    for iteration in range(n_iter):
        grad = np.zeros_like(F)
        for i, u in enumerate(nodes):
            f_u = F[i]
            # Neighbor contribution
            for v in adj[u]:
                j = idx[v]
                dot = f_u @ F[j]
                if dot > 0:
                    factor = math.exp(-dot) / (1 - math.exp(-dot) + 1e-10)
                    grad[i] += factor * F[j]
            # Non-neighbor contribution (use col_sums trick)
            grad[i] -= (col_sums - F[i])
        # Update
        old_F = F.copy()
        F += lr * grad
        F = np.maximum(F, 1e-6)    # non-negativity
        col_sums = F.sum(axis=0)
        # Early stopping
        if np.linalg.norm(F - old_F) < 1e-4: break

    return nodes, F

# Build a graph with two overlapping communities
adj = defaultdict(set)
edges = [(0,1),(1,2),(2,0),(2,3),(3,4),(4,5),(5,3),(2,5)]
for u,v in edges:
    adj[u].add(v); adj[v].add(u)
for i in range(6): adj.setdefault(i,set())

nodes, F = bigclam(adj, k=2, n_iter=200, lr=0.005)
print("Node memberships (F matrix):")
print(f"{'Node':>6} {'C1':>8} {'C2':>8} {'Primary':>10}")
for n, f in zip(nodes, F):
    primary = 'C1' if f[0] > f[1] else 'C2'
    print(f"{n:>6} {f[0]:>8.3f} {f[1]:>8.3f} {primary:>10}")
print(f"
Node 2 and 5 should show high membership in both communities (they bridge)")

## Detecting Overlapping Nodes

A node with significant membership in multiple communities is a **bridge node**.

**Detection:** node $u$ belongs to community $c$ if $F_{uc} > \delta$ (threshold).

**Community assignment:**
- Hard: assign to community with maximum $F_{uc}$
- Soft: assign to all communities with $F_{uc} > \delta$

In [ ]:
# Soft community assignment
delta = 0.2
print("Community memberships (soft assignment, threshold=0.2):")
for n, f in zip(nodes, F):
    comms = [f'C{c+1}' for c in range(f.shape[0]) if f[c] > delta]
    print(f"  Node {n}: {comms if comms else ['(none)']}")

## Comparison: Overlapping vs Hard Partition

| Method | Community type | Handles overlaps | Scalability | Interpretability |
|--------|---------------|-----------------|-------------|-----------------|
| Louvain | Hard | No | Very good | Medium |
| Spectral | Hard | No | Good | Low |
| BigCLAM | Soft | **Yes** | Good | High |
| SLPA | Soft | **Yes** | Good | Medium |

**When to use overlapping:**
- Domain knowledge suggests nodes have multiple roles
- Hard partitions show many nodes on community boundaries
- Downstream task benefits from fractional memberships (recommendation, routing)